# Notebook Title

## Overfit & Debordes

### Simon Simus et Nathan Vidal

November 2025

#-----------------------------------------------------------------------------------------------------------------------------------

For more details in our work (EDA, Training, hyperparameters optimizaiton), **check our github repo** : https://github.com/Simon773/ML_Challenge

#-------------------------------------------------------------------------------------------------------------------------------------

This notebook presents the results of our work for the ML Challenge. It is structured as follows:
- Data Wrangling
- Results of the final model and exportation for test datasets. 
- Analyses of the data and the models (plots of the presentation)

#### Packages importation

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone as sklearn_clone #clone models for cross-validation
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

## Data Wrangling

### Data importation

In [6]:
# Adjust the path if needed
X_train = pd.read_csv("data/challenge_train_features.csv", index_col=0)
y_train = pd.read_csv("data/challenge_train_revenue.csv", index_col=0)
X_test = pd.read_csv("data/challenge_test_features.csv", index_col=0)

Or directly with the website link

In [7]:

# X_train = pd.read_csv("https://edouardpauwels.fr/MLM2DSSS/challenge_train_features.csv",index_col=0)
# y_train = pd.read_csv("https://edouardpauwels.fr/MLM2DSSS/challenge_train_revenue.csv",index_col=0)
# X_test = pd.read_csv("https://edouardpauwels.fr/MLM2DSSS/challenge_test_features.csv",index_col=0)

## Modeling and exportation

### Preprocessing data


In this part, we process the data and fit the final model on the training dataset. 

In [8]:
# ----------------------------------------------------------------
# Preprocessing functions
# ----------------------------------------------------------------

# Preprocessing functions


# Create a binary indicator for zero budget (nan values)
def budget_missing_indicator(X):
    X = X.copy()
    X["budget_is_zero"] = (X["budget"] == 0).astype(int)
    return X


def log_budget(X):
    "convert budget in log"
    X = X.copy()
    X["budget"] = np.log1p(X["budget"].clip(lower=0))
    return X


def log_popularity(X):
    "convert popularity score in log"
    X = X.copy()
    X["popularity_score"] = np.log1p(X["popularity_score"].clip(lower=0))
    return X


def budget_0_to_nan(X):
    "replace zero budget with nan in a new column : budget_nonzero"
    X = X.copy()
    X["budget_nonzero"] = X["budget"].replace(0, np.nan)
    return X


def collection_to_binary(X):
    "create a binary column has_collection where 1 indicates the movie belongs to a collection"
    X = X.copy()
    X["has_collection"] = X["collection"].notna().astype(int)
    return X


def english_to_binary(X):
    """
    Convert language column to binary where:
    1 = movie is in English ('en')
    0 = movie is in another language
    """
    X = X.copy()
    X["language"] = (X["language"] == "en").astype(int)
    return X


def count_countries(X):
    """create a column 'country_count' that counts the number of countries listed
    in the 'country' column.
    """
    X = X.copy()
    X["country_count"] = (
        X["country"]
        .fillna("")
        .apply(lambda val: len(str(val).split(",")) if str(val).strip() != "" else 0)
    )
    return X


def get_main_countries(X, top_n=7):
    """
    return N top countries based on frequency in the country column.
    The main country is the first in the country list.
    """
    #extract main country
    country_main = (
        X["country"]
        .apply(lambda x: x.split(",")[0] if isinstance(x, str) else "Unknown")
        .astype(str)
        .str.strip()
    )
    top_countries = country_main.value_counts().nlargest(top_n).index.tolist()

    return top_countries


def add_country_dummies(X, top_countries):
    """
    Create dummies for main countries present in top_countries.
    Countries not present are grouped into Other.
    """
    X = X.copy()
    X["country_main"] = (
        X["country"]
        .apply(lambda x: x.split(",")[0] if isinstance(x, str) else "Unknown")
        .astype(str)
        .str.strip()
    )

    # set other countries to Other
    X["country_main"] = X["country_main"].apply(
        lambda c: c if c in top_countries else "Other"
    )
    dummies = pd.get_dummies(X["country_main"], prefix="country", dtype=int)
    X = pd.concat([X, dummies], axis=1)

    return X, list(dummies.columns) #return also the list of created columns to add later to features list


def add_top_company_features(X, top_revenue_companies):
    """
    Adds two columns:
    num_top_companies : number of the movie's studios present in the top_revenue_companies list
    is_top_company: 1 if at least one of the movie's studios is in the list, otherwise 0
    """
    X = X.copy()

    def count_star_companies(company_str):
        if not isinstance(company_str, str) or company_str.strip() == "":
            return 0
        companies = [c.strip() for c in company_str.split(",")]
        return sum(c in top_revenue_companies for c in companies)

    # Number of prestigious studios in each movie
    X["num_top_companies"] = X["company"].fillna("").apply(count_star_companies)

    # Binary variable if at least 1 big studio
    X["is_top_company"] = (X["num_top_companies"] > 0).astype(int)
    return X


def get_frequent_genres(X, min_occurrence=150):
    """
    Get list of genres that appear more than min_occurrence times
    return a list of frequent genres + "Other"
    """
    X = X.copy()
    X["genres_list"] = X["genre"].apply(
        lambda x: x.split(",") if isinstance(x, str) else []
    )
    genre_counts = X["genres_list"].explode().value_counts()
    selected_genres = genre_counts[genre_counts >= min_occurrence].index.tolist()
    return selected_genres + ["Other"]


def encode_genres(X, selected_genres):
    """
    Encode genres into binary columns based on a predefined list of genres
    """
    X = X.copy()
    X["genres_list"] = X["genre"].apply(
        lambda x: x.split(",") if isinstance(x, str) else []
    )

    # Map less frequent genres to "Other"
    X["genres_list"] = X["genres_list"].apply(
        lambda lst: [g if g in selected_genres else "Other" for g in lst]
    )

    # Create binary columns for each genre
    for genre in selected_genres:
        X[genre] = X["genres_list"].apply(lambda lst: int(genre in lst))

    return X


def add_star_actor_features(X, star_actors):
    """
    Adds two columns:
    num_stars: number of prestigious actors present in the movie's cast
    has_stars: 1 if at least one prestigious actor is present
    """
    X = X.copy()

    def count_stars(actor_str):
        if not isinstance(actor_str, str) or actor_str.strip() == "":
            return 0
        actors = [a.strip() for a in actor_str.split(",")]
        return sum(a in star_actors for a in actors)

    # Number of prestigious actors in each movie
    X["num_stars"] = X["cast"].fillna("").apply(count_stars)

    # Presence or absence of at least one prestigious actor
    X["has_stars"] = (X["num_stars"] > 0).astype(int)

    return X


def extract_release_year(X, date_column="date"):
    """
    Extract release year from date column and correct future years
    """
    X = X.copy()
    X["date_format"] = pd.to_datetime(
        X[date_column], format="%m/%d/%y", errors="coerce"
    )
    # correct issue with years > 2025
    mask_future = X["date_format"].dt.year > 2025
    X.loc[mask_future, "date_format"] -= pd.offsets.DateOffset(years=100)
    X["release_year"] = X["date_format"].dt.year

    return X

In [9]:
def preprocess_data(X_train, X_test, y_train):
    """
    We use this function to apply previous preprocessing functions on both train and test sets
    """

    X_train = budget_missing_indicator(X_train)
    X_test = budget_missing_indicator(X_test)

    X_train = budget_0_to_nan(X_train)
    X_test = budget_0_to_nan(X_test)

    X_train = collection_to_binary(X_train)
    X_test = collection_to_binary(X_test)

    selected_genres = get_frequent_genres(X_train, min_occurrence=150)
    X_train = encode_genres(X_train, selected_genres)
    X_test = encode_genres(X_test, selected_genres)

    X_train = count_countries(X_train)
    X_test = count_countries(X_test)

    top_countries = get_main_countries(X_train, top_n=6)
    X_train, created_cols = add_country_dummies(X_train, top_countries)
    X_test, _ = add_country_dummies(X_test, top_countries)

    X_train = log_popularity(X_train)
    X_test = log_popularity(X_test)

    X_train = english_to_binary(X_train)
    X_test = english_to_binary(X_test)

    # Interaction features
    X_train["budget_x_popularity"] = (
        X_train["budget_nonzero"] * X_train["popularity_score"]
    )
    X_test["budget_x_popularity"] = (
        X_test["budget_nonzero"] * X_test["popularity_score"]
    )

    X_train["budget_nonzero_length_ratio"] = X_train["budget_nonzero"] / X_train[
        "length"
    ].replace(0, np.nan)
    X_test["budget_nonzero_length_ratio"] = X_test["budget_nonzero"] / X_test[
        "length"
    ].replace(0, np.nan)

    X_train["length_x_popularity"] = X_train["length"] * X_train["popularity_score"]
    X_test["length_x_popularity"] = X_test["length"] * X_test["popularity_score"]

    X_train = extract_release_year(X_train)
    X_test = extract_release_year(X_test)

    top_revenue_companies = [
        "Walt Disney Pictures",
        "Marvel Studios",
        "Lucasfilm",
        "20th Century Studios",
        "Warner Bros. Pictures",
        "Universal Pictures",
        "Paramount Pictures",
        "Columbia Pictures",
        "Legendary Entertainment",
        "Pixar Animation Studios",
    ]

    X_train = add_top_company_features(X_train, top_revenue_companies)
    X_test = add_top_company_features(X_test, top_revenue_companies)

    star_actors = [
        "Charlie Chaplin",
        "Clark Gable",
        "Humphrey Bogart",
        "James Stewart",
        "Cary Grant",
        "Marlon Brando",
        "Robert De Niro",
        "Al Pacino",
        "Jack Nicholson",
        "Clint Eastwood",
        "Harrison Ford",
        "Sylvester Stallone",
        "Arnold Schwarzenegger",
        "Tom Hanks",
        "Tom Cruise",
        "Leonardo DiCaprio",
        "Brad Pitt",
        "Johnny Depp",
        "Will Smith",
        "Denzel Washington",
        "George Clooney",
        "Robert Downey Jr.",
        "Chris Hemsworth",
        "Ryan Reynolds",
        "Joaquin Phoenix",
        "Adam Driver",
        "Timothée Chalamet",
        "Chris Evans",
        "Mark Ruffalo",
        "Samuel L. Jackson",
        "Marilyn Monroe",
        "Audrey Hepburn",
        "Katharine Hepburn",
        "Grace Kelly",
        "Elizabeth Taylor",
        "Ingrid Bergman",
        "Meryl Streep",
        "Jane Fonda",
        "Jodie Foster",
        "Sigourney Weaver",
        "Julia Roberts",
        "Nicole Kidman",
        "Cate Blanchett",
        "Sandra Bullock",
        "Cameron Diaz",
        "Angelina Jolie",
        "Charlize Theron",
        "Scarlett Johansson",
        "Jennifer Lawrence",
        "Emma Stone",
    ]
    X_train = add_star_actor_features(X_train, star_actors)
    X_test = add_star_actor_features(X_test, star_actors)

    # feature list
    feature_columns = (
        [
            "popularity_score",
            "budget_nonzero",
            "budget_is_zero",
            "has_collection",
            "language",
            "length",
            "country_count",
            "is_top_company",
            "has_stars",
            "budget_nonzero_length_ratio",
            "release_year",
            # "budget_x_popularity",
            # "length_x_popularity",  #see if we keep it
        ]
        + selected_genres #for genre dummies
        + created_cols #for country dummies 
    )

    X_train_final = X_train[feature_columns]
    X_test_final = X_test[feature_columns]

    # Transform target variable with log base 10 in order to minimize MSLE
    y_train_log = np.log10(1 + y_train)

    return X_train_final, X_test_final, y_train_log


### Model Training

In [14]:
def train_and_get_predictions(X_train, y_train, X_test,filename="predictions.txt"):
    """train catboost model and get predictions on test set
    """
    
    # preprocess data
    X_train_processed, X_test_processed, y_train_log = preprocess_data(
            X_train, X_test, y_train
        )

    #initialize and train catBoost model
    train_pool = Pool(X_train_processed, y_train_log)
    model = CatBoostRegressor(
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        l2_leaf_reg=8,
        random_seed=42,
        verbose=100,
        early_stopping_rounds=50,
    )
    model.fit(train_pool)

    # make predictions on test set
    y_pred = model.predict(X_test_processed)
    # inverse transform predictions (from log base 10scale to original scale)
    y_pred = (10**y_pred) - 1

    #save predictions in txt file that will use in ML challenge submission
    pred_str = ",".join([str(int(p)) for p in y_pred])
    with open(filename, "w") as f:
        f.write(pred_str)

#try our model and get predictions on test set 
train_and_get_predictions(X_train, y_train, X_test)

0:	learn: 1.2373094	total: 3.58ms	remaining: 3.58s
100:	learn: 0.8015139	total: 161ms	remaining: 1.43s
200:	learn: 0.7581779	total: 324ms	remaining: 1.29s
300:	learn: 0.7235389	total: 463ms	remaining: 1.07s
400:	learn: 0.6780259	total: 603ms	remaining: 900ms
500:	learn: 0.6452000	total: 757ms	remaining: 754ms
600:	learn: 0.6154170	total: 893ms	remaining: 593ms
700:	learn: 0.5891636	total: 1.04s	remaining: 444ms
800:	learn: 0.5653275	total: 1.18s	remaining: 292ms
900:	learn: 0.5422408	total: 1.31s	remaining: 144ms
999:	learn: 0.5237571	total: 1.46s	remaining: 0us


In this part, we can predict the revenues on the new test datasets and export the results as a csv file. 

### On a new secret test set

In [ ]:
X_final_test = pd.read_csv("data/challenge_test_features.csv", index_col=0)
#to be replaced by 
# X_final_test = pd.read_csv("path_of_your_csv_file",index_col=0)

#try our model and get predictions on the secret test set
train_and_get_predictions(X_train, y_train, X_final_test,filename="final_predictions.txt")

#get your predictions in final_predictions.txt file

0:	learn: 1.2373094	total: 1.71ms	remaining: 1.7s
100:	learn: 0.8015139	total: 148ms	remaining: 1.31s
200:	learn: 0.7581779	total: 288ms	remaining: 1.15s
300:	learn: 0.7235389	total: 422ms	remaining: 981ms
400:	learn: 0.6780259	total: 559ms	remaining: 835ms
500:	learn: 0.6452000	total: 696ms	remaining: 693ms
600:	learn: 0.6154170	total: 838ms	remaining: 556ms
700:	learn: 0.5891636	total: 975ms	remaining: 416ms
800:	learn: 0.5653275	total: 1.11s	remaining: 277ms
900:	learn: 0.5422408	total: 1.25s	remaining: 137ms
999:	learn: 0.5237571	total: 1.39s	remaining: 0us


## Other Analyses


### Compare models in order to find the best with cross validation

In [17]:
# NATHAN IL FAUT QUE TU AJOUTES TA REGRESSION : OUBLIE PAS DE GERER LES NAN DANS LA REGRESSION


def compare_models(X, y):
    y = y.values.ravel()
    models = {
        "CatBoost": CatBoostRegressor(
            learning_rate=0.05,
            depth=6,
            loss_function="RMSE",
            l2_leaf_reg=8,
            random_seed=2,
            verbose=0,
            early_stopping_rounds=50,
        ),
        "XGBoost": XGBRegressor(
            n_estimators=3000,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            min_child_weight=1.0,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=2,
            n_jobs=-1,
            early_stopping_rounds=50,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=100, random_state=2, n_jobs=-1
        ),
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    for name, model_template in models.items():
        fold_scores = []
        #train 4 models with cross validation
        for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):

            model = sklearn_clone(model_template)

            #create train and validation sets with CV folds
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            if name == "CatBoost":
                model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=0)

            elif name == "XGBoost":
                model.fit(
                    X_train,
                    y_train,
                    eval_set=[(X_val, y_val)],
                    verbose=0,
                )

            elif name == "RandomForest":
                model.fit(X_train, y_train)


            y_pred = model.predict(X_val)
            y_pred = y_pred.clip(0, None) #ensure no negative predictions
            score = mean_squared_error(y_val, y_pred)
            fold_scores.append(score)

        mean_score = np.mean(fold_scores)
        print(f"{name} MSLE mean: {mean_score} over 5 folds")

X_train_processed, X_test_processed, y_train_log = preprocess_data(
            X_train, X_test, y_train
        )
compare_models(X_train_processed, y_train_log)

CatBoost MSLE mean: 0.7306476626005557 over 5 folds
XGBoost MSLE mean: 0.7374925853596109 over 5 folds
RandomForest MSLE mean: 0.7611059101805713 over 5 folds


We can see thaht Catboost have the best MSLE in mean over the 5 folds. We will use this model and we will realize an optimisation of hyperparameter

### **Important if you want to run these cell, you need to have a wandb account and add your API key**

like this in .env file : WANDB_API_KEY=fkskfshfks

In [ ]:
from dotenv import load_dotenv
import os
import wandb

# ------------------ init W&B ------------------
load_dotenv()
api_key = os.getenv("WANDB_API_KEY")
wandb.login(key=api_key)
PROJECT_NAME = "film-revenue-catboost-tuning"

# ------------------ data + preprocessing ------------------
X = pd.read_csv("data/challenge_train_features.csv", index_col=0)
y = pd.read_csv("data/challenge_train_revenue.csv", index_col=0).iloc[:, 0]
X_test = pd.read_csv("data/challenge_test_features.csv", index_col=0)

X_proc, X_test_proc, y_log = preprocess_data(X, X_test, y)


# ------------------ fonction W&B K-Fold ------------------
def train_and_log(config=None):
    with wandb.init(config=config, project=PROJECT_NAME):
        cfg = wandb.config

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X_proc)):
            X_train_f = X_proc.iloc[train_idx]
            y_train_f = y_log.iloc[train_idx]

            X_val_f = X_proc.iloc[val_idx]
            y_val_f = y_log.iloc[val_idx]

            model = CatBoostRegressor(
                learning_rate=cfg.learning_rate,
                depth=cfg.depth,
                l2_leaf_reg=cfg.l2_leaf_reg,
                loss_function="RMSE",
                random_seed=42,
                early_stopping_rounds=50,
                verbose=False,
            )

            model.fit(X_train_f, y_train_f, eval_set=(X_val_f, y_val_f), verbose=False)

            y_val_pred_log = model.predict(X_val_f)

            score = mean_squared_error(y_val_f, y_val_pred_log)
            fold_scores.append(score)

        # mean score over 5 folds
        mean_score = np.mean(fold_scores)

        wandb.log({"msle": mean_score})
        print(f" CV MSLE = {mean_score}")


# ------------------ sweep configuraion ------------------
sweep_config = {
    "method": "bayes",
    "metric": {"name": "msle", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"distribution": "uniform", "min": 0.02, "max": 0.10},
        "depth": {"values": [4, 5, 6, 7]},
        "l2_leaf_reg": {"values": [1, 3, 6, 10]},
    },
}



sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, function=train_and_log, count=2)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\simus\_netrc
wandb: Currently logged in as: simon-s (simon-s-toulouse-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Create sweep with ID: lspj2byx
Sweep URL: https://wandb.ai/simon-s-toulouse-school-of-economics/film-revenue-catboost-tuning/sweeps/lspj2byx


wandb: Agent Starting Run: i8aqo1v7 with config:
wandb: 	depth: 4
wandb: 	l2_leaf_reg: 1
wandb: 	learning_rate: 0.0762098381644353


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


 CV MSLE = 0.7189051633981848


msle,▁
msle,0.71891
